# Homework 2 — CNNs from Scratch on CIFAR-10 + Optuna

**Starting notebook to complete:**  
https://colab.research.google.com/github/serivan/DeepLearning/blob/master/Lessons/05_cnn_ch12_live_skeleton.ipynb

This homework has two parts:

1. **Complete the live CNN notebook** reproduced below: GPU use, convolutions, tensor shapes, a Fashion-MNIST CNN, training, evaluation, and the ablation activity.
2. **CIFAR-10 challenge:** design a stronger CNN **from scratch** (no transfer learning), test techniques studied in class including **data augmentation**, and tune the most relevant hyperparameters systematically with **Optuna**.

### Submission expectations
- Keep the important intermediate outputs and plots in the notebook.
- State a short hypothesis before each comparison and comment on the result afterwards.
- Use the **validation set** for model/hyperparameter choices.
- Use the **test set only once**, after the final configuration has been selected.
- Save the final trained model checkpoint.

> **No transfer learning is allowed in H2.** Transfer learning is addressed separately in H3.


# Part A — Complete the live CNN notebook

Complete all TODOs in the following material. Do not skip the reasoning questions: shape calculations, device placement, and error analysis are part of the assignment.


In [ ]:
import time
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

SEED = 42
torch.manual_seed(SEED)

# Colab: Runtime > Change runtime type > T4 GPU (if available)
device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

print("PyTorch:", torch.__version__)
print("device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))


## 1. CPU → GPU: what does it really mean? **[LIVE]**

A tensor lives on a *device*. Operations require data and model parameters to be on the same device.

### Predict before running
What do you expect from these three operations?
- create `A` and `B` on the CPU;
- copy them to the GPU with `.to(device)`;
- multiply them with `A @ B`.

For meaningful CUDA benchmarking, two details matter:
- **warm-up**, because the first operation includes initialization overhead;
- `torch.cuda.synchronize()` before stopping the timer, because CUDA kernels execute asynchronously.


In [ ]:
N = 4096
A_cpu = torch.randn(N, N)
B_cpu = torch.randn(N, N)

# TODO 1 — measure the CPU matrix-multiplication time
# Suggerimento: time.perf_counter()

if device.type == "cuda":
    # TODO 2 — move A_cpu and B_cpu to the GPU and measure transfer time

    # TODO 3 — run ONE warm-up multiplication and synchronize CUDA

    # TODO 4 — measure A_gpu @ B_gpu, remembering torch.cuda.synchronize()

    # TODO 5 — copy the result back to CPU and compare it with the CPU result
    pass
else:
    print("No CUDA GPU: continuing on", device)


### ACTIVITY 1 — transfer or computation?

Predict what happens if `N` becomes much smaller, for example `N=256`.

**Question:** is the GPU still necessarily faster *end-to-end*? Why can transfer cost dominate?

> Key message: a GPU is most useful when we perform **a lot of computation per byte transferred** and, during training, keep both model and mini-batches on the GPU instead of repeatedly moving data CPU↔GPU.


## 2. Dataset and mini-batches: inspect before modeling **[LIVE]**

We will use **Fashion-MNIST**: 28×28 grayscale images, 10 classes. It is small enough for live training but non-trivial enough to show why CNNs are useful.

Before running the cell: what shape do you expect for a mini-batch of 128 images?


In [ ]:
transform = transforms.ToTensor()

train_full = datasets.FashionMNIST("data", train=True, download=True, transform=transform)
test_data = datasets.FashionMNIST("data", train=False, download=True, transform=transform)

# Keep a separate validation split.
g = torch.Generator().manual_seed(SEED)
train_data, valid_data = torch.utils.data.random_split(train_full, [55_000, 5_000], generator=g)

# FAST_MODE keeps the live session practical and responsive.
FAST_MODE = True
if FAST_MODE:
    train_data = Subset(train_data, range(12_000))
    valid_data = Subset(valid_data, range(2_000))
    test_live = Subset(test_data, range(2_000))
else:
    test_live = test_data

BATCH_SIZE = 128
pin = device.type == "cuda"
train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True, pin_memory=pin)
valid_loader = DataLoader(valid_data, batch_size=BATCH_SIZE, shuffle=False, pin_memory=pin)
test_loader = DataLoader(test_live, batch_size=BATCH_SIZE, shuffle=False, pin_memory=pin)

class_names = train_full.classes
X, y = next(iter(train_loader))
print("X:", X.shape, X.dtype, "range:", (X.min().item(), X.max().item()))
print("y:", y.shape, y.dtype)
print(class_names)

fig, axes = plt.subplots(1, 8, figsize=(12, 2))
for ax, img, label in zip(axes, X[:8], y[:8]):
    ax.imshow(img.squeeze(0), cmap="gray")
    ax.set_title(class_names[label])
    ax.axis("off")
plt.tight_layout()


## 3. A convolution before the CNN **[LIVE]**

Before using `nn.Conv2d`, let us make the operation visible. We create a synthetic image with a vertical edge and apply a kernel that measures horizontal intensity differences.

For an input `[B, C, H, W]`, `F.conv2d` expects a kernel with shape `[C_out, C_in, kH, kW]`.


In [ ]:
img = torch.zeros(1, 1, 12, 12)
img[:, :, :, 6:] = 1.0

# TODO 6 — build a 3x3 kernel sensitive to a vertical edge.
# Partenza suggerita:
# [-1, 0, 1]
# [-1, 0, 1]
# [-1, 0, 1]
# Ricorda poi di ottenere shape [1, 1, 3, 3].

# TODO 7 — apply F.conv2d(..., padding=1)

# TODO 8 — visualizza input, kernel e feature map


## 4. Shape reasoning: padding, stride, pooling **[ACTIVITY]**

For one spatial dimension:

\[
H_{out}=\left\lfloor\frac{H+2P-K}{S}\right\rfloor+1
\]

### ACTIVITY 2 — paper and pencil
For an input of size `28×28`, compute the shapes after:
1. `Conv2d(1, 16, kernel_size=3, padding=1)`;
2. `MaxPool2d(2)`;
3. `Conv2d(16, 32, kernel_size=3, padding=1)`;
4. `MaxPool2d(2)`.

How many features enter the first linear layer?


In [ ]:
probe = torch.randn(4, 1, 28, 28)
blocks = nn.Sequential(
    nn.Conv2d(1, 16, kernel_size=3, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(2),
    nn.Conv2d(16, 32, kernel_size=3, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(2),
)

z = probe
for layer in blocks:
    z = layer(z)
    print(f"{layer.__class__.__name__:12s} -> {tuple(z.shape)}")
print("flatten features/image:", z[0].numel())


## 5. Build the CNN **[LIVE]**

Compact architecture:

`[1,28,28] → Conv(16) → Pool → Conv(32) → Pool → Flatten → 64 → 10`

Questions to ask before completing the code:
- why is the first `in_channels` equal to 1?
- what does `out_channels` represent?
- why do we **not** add Softmax at the output?
- how many parameters does a 3×3 convolution use compared with an equivalent fully connected layer?


In [ ]:
class SmallCNN(nn.Module):
    def __init__(self):
        super().__init__()

        # TODO 9 — features:
        # Conv2d(1,16,3,padding=1) -> ReLU -> MaxPool2d(2)
        # Conv2d(16,32,3,padding=1) -> ReLU -> MaxPool2d(2)
        self.features = nn.Sequential(
            # ...
        )

        # TODO 10 — classifier: Flatten -> Linear(? ,64) -> ReLU -> Linear(64,10)
        self.classifier = nn.Sequential(
            # ...
        )

    def forward(self, x):
        # TODO 11 — forward pass
        pass

# TODO 12 — instantiate the model and move it to device
# model = ...

# Check: for 8 images, the output must have shape [8, 10]


## 6. Device-aware training loop **[LIVE]**

With `CrossEntropyLoss`, the model returns **logits**. Each mini-batch follows:

`zero_grad → forward → loss → backward → optimizer.step`.

The important GPU point is to move **each mini-batch** to `device`, while keeping the model there throughout training.


In [ ]:
def run_epoch(model, loader, criterion, optimizer=None):
    training = optimizer is not None
    model.train(training)
    total_loss = 0.0
    total_correct = 0
    total_examples = 0

    for X, y in loader:
        # TODO 13 — move X and y to device (non_blocking=True se vuoi sfruttare pin_memory)

        if training:
            # TODO 14 — zero the gradients
            pass

        with torch.set_grad_enabled(training):
            # TODO 15 — forward + loss

            if training:
                # TODO 16 — backward + optimizer step
                pass

        # TODO 17 — accumulate loss, correct predictions, and number of examples

    # TODO 18 — return mean loss and accuracy
    raise NotImplementedError

# TODO 19 — loss and optimizer
# criterion = ...
# optimizer = ...


In [ ]:
EPOCHS = 3 if FAST_MODE else 5
history = []

# TODO 20 — for each epoch:
# 1) run_epoch(..., optimizer) sul train
# 2) run_epoch(...) sulla validation
# 3) stampa loss/accuracy e tempo dell'epoca


## 7. Evaluation: more than a single accuracy value **[LIVE]**

Inspect a few misclassified examples. They are often more informative than a single metric.


In [ ]:
# TODO 21 — evaluate the model on the test set

# TODO 22 — find some misclassified images and visualize:
# image, true class, predicted class

wrong = (pred != y).nonzero(as_tuple=True)[0][:8]
fig, axes = plt.subplots(1, len(wrong), figsize=(14, 2.4))
if len(wrong) == 1:
    axes = [axes]
for ax, i in zip(axes, wrong):
    ax.imshow(X[i, 0], cmap="gray")
    ax.set_title(f"true: {class_names[y[i]]}\npred: {class_names[pred[i]]}")
    ax.axis("off")
plt.tight_layout()

## 8. ACTIVITY 3 — ablation challenge **[ACTIVITY]**

Work individually or in pairs. Choose **one** modification, state a hypothesis first, then test it:

- **A.** replace the second `MaxPool2d(2)` with `Identity` and fix the input size of the `Linear` layer;
- **B.** add `BatchNorm2d` after the convolutional layers;
- **C.** add `Dropout(0.3)` to the classifier;
- **D.** double the channels from `16→32` and `32→64`.

Record three things: **number of parameters**, **training cost**, and **validation accuracy**.
The goal is not to find a single “winner”, but to explain **why** the change affects capacity, computational cost, and generalization.


# Part B — CIFAR-10: build a stronger CNN from scratch

Fashion-MNIST is grayscale and relatively simple. CIFAR-10 contains **32×32 RGB images** from 10 classes and is a more demanding test for a CNN trained from scratch.

Your goal is not merely to obtain the highest possible score. You must design a sensible experiment, compare techniques, and explain why the selected configuration performs better.

You should consider the techniques studied so far, including:
- increasing/decreasing convolutional capacity;
- `BatchNorm2d`;
- dropout;
- activation function;
- SGD vs AdamW;
- learning rate, weight decay, and momentum;
- **dataset augmentation**;
- early stopping.


In [ ]:
# Extra imports used in Part B
import copy
import random
import numpy as np
import pandas as pd

random.seed(SEED)
np.random.seed(SEED)


## B1 — CIFAR-10 data and augmentation

Use exactly the same train/validation indices for all experiments, so that comparisons remain fair.

Start with a standard CIFAR-10 normalization. Then define a training augmentation pipeline. A common starting point is:

- random crop to 32×32 with padding;
- random horizontal flip;
- tensor conversion and normalization.

You may add **one additional reasonable augmentation** (for example mild color jitter), but avoid transformations that destroy the class semantics.

**Question before coding:** why should augmentation be applied only to the training subset and not to validation/test data?


In [ ]:
CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD = (0.2470, 0.2435, 0.2616)

plain_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
])

# TODO B1 — define an augmented training transform.
# Minimum required augmentation:
#   RandomCrop(32, padding=4)
#   RandomHorizontalFlip()
# followed by ToTensor() and Normalize(...).
train_transform_aug = transforms.Compose([
    # ...
])

# Two views of the same CIFAR-10 training examples:
# one deterministic, one augmented.
cifar_train_plain_full = datasets.CIFAR10(
    "data", train=True, download=True, transform=plain_transform
)
cifar_train_aug_full = datasets.CIFAR10(
    "data", train=True, download=True, transform=train_transform_aug
)
cifar_test = datasets.CIFAR10(
    "data", train=False, download=True, transform=plain_transform
)

# Fixed split: the validation examples are never augmented.
g = torch.Generator().manual_seed(SEED)
perm = torch.randperm(len(cifar_train_plain_full), generator=g)
valid_idx = perm[:5_000].tolist()
train_idx = perm[5_000:].tolist()

# Optional classroom/homework speed switch. The final model will use all data.
HW_FAST_MODE = True
if HW_FAST_MODE:
    train_idx_work = train_idx[:20_000]
else:
    train_idx_work = train_idx

cifar_train_plain = Subset(cifar_train_plain_full, train_idx_work)
cifar_train_aug = Subset(cifar_train_aug_full, train_idx_work)
cifar_valid = Subset(cifar_train_plain_full, valid_idx)

CIFAR_BATCH_SIZE = 128
cifar_train_plain_loader = DataLoader(
    cifar_train_plain, batch_size=CIFAR_BATCH_SIZE, shuffle=True,
    pin_memory=pin
)
cifar_train_aug_loader = DataLoader(
    cifar_train_aug, batch_size=CIFAR_BATCH_SIZE, shuffle=True,
    pin_memory=pin
)
cifar_valid_loader = DataLoader(
    cifar_valid, batch_size=CIFAR_BATCH_SIZE, shuffle=False,
    pin_memory=pin
)
cifar_test_loader = DataLoader(
    cifar_test, batch_size=CIFAR_BATCH_SIZE, shuffle=False,
    pin_memory=pin
)

# TODO B2 — inspect one batch and visualize at least 8 augmented images.
# Confirm the expected shape [B, 3, 32, 32].


## B2 — Design a better CIFAR-10 CNN

Design your own CNN **from scratch**. You may start from the Fashion-MNIST architecture, but the final network must be meaningfully improved for CIFAR-10.

### Minimum requirements
Your model must:
- accept `[B, 3, 32, 32]` inputs;
- contain at least **three convolutional stages**;
- reduce spatial resolution progressively;
- use at least **two** techniques among `BatchNorm2d`, dropout, increased channel capacity, alternative activation, or global/adaptive pooling;
- return **10 logits**;
- expose at least three parameters that Optuna can later vary.

A useful interface is provided below so that the same class can be reused during tuning. You are free to improve the internal architecture.


In [ ]:
def make_activation(name):
    if name == "relu":
        return nn.ReLU()
    if name == "gelu":
        return nn.GELU()
    raise ValueError(f"Unknown activation: {name}")


class CIFAR10CNN(nn.Module):
    def __init__(
        self,
        base_channels=32,
        dropout=0.3,
        use_batchnorm=True,
        activation="relu",
        n_classes=10,
    ):
        super().__init__()

        # TODO B3 — implement your CNN.
        # Suggested channel progression: C -> 2C -> 4C.
        # You may use Conv -> BatchNorm (optional) -> activation -> pooling.
        # AdaptiveAvgPool2d(1) can make the classifier independent of
        # the exact spatial size before the final head.
        self.features = nn.Sequential(
            # ...
        )

        self.classifier = nn.Sequential(
            # ...
        )

    def forward(self, x):
        # TODO B4 — complete the forward pass.
        pass


# TODO B5 — instantiate a baseline configuration and verify:
# 1) output shape for a synthetic [8, 3, 32, 32] batch;
# 2) total number of parameters.


## B3 — Controlled experiment: does augmentation help?

Before running Optuna, isolate the effect of augmentation.

Train **the same architecture with the same initialization and optimizer settings** twice:

1. without augmentation;
2. with your augmentation pipeline.

Keep the number of epochs small and identical. Report training/validation accuracy and discuss whether augmentation changes the training/validation gap.

> A lower training accuracy together with a higher validation accuracy can be a perfectly reasonable outcome when augmentation makes the training task harder but improves generalization.


In [ ]:
def fit_fixed_epochs(model, train_loader, valid_loader, criterion, optimizer, epochs=4):
    history = []
    best_state = copy.deepcopy(model.state_dict())
    best_val_acc = -1.0

    for epoch in range(epochs):
        train_loss, train_acc = run_epoch(model, train_loader, criterion, optimizer)
        val_loss, val_acc = run_epoch(model, valid_loader, criterion)
        history.append({
            "epoch": epoch + 1,
            "train_loss": train_loss,
            "train_acc": train_acc,
            "val_loss": val_loss,
            "val_acc": val_acc,
        })
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state = copy.deepcopy(model.state_dict())

        print(
            f"epoch {epoch+1:02d} | "
            f"train acc {train_acc:.3%} | val acc {val_acc:.3%}"
        )

    model.load_state_dict(best_state)
    return pd.DataFrame(history), best_val_acc


# TODO B6 — define one fixed baseline configuration.
# Use the SAME random seed before creating each model.
# Train once with cifar_train_plain_loader and once with cifar_train_aug_loader.
# Store the two best validation accuracies and compare them.


### B3 reflection

Write a short answer directly in the notebook:

1. What changed when augmentation was introduced?
2. Did the training/validation gap increase or decrease?
3. Which augmentations are plausible for CIFAR-10 and which could be harmful?
4. Based on this experiment, should augmentation be part of the Optuna search?


## B4 — Systematic hyperparameter tuning with Optuna

Now use Optuna to search a **small but meaningful** space. The goal is to analyze hyperparameter behavior, not to launch hundreds of trials.

Your search must include at least:
- base channel width;
- dropout;
- BatchNorm on/off;
- ReLU vs GELU;
- augmentation on/off;
- optimizer: AdamW vs SGD;
- learning rate;
- weight decay;
- momentum **only when SGD is selected**.

Use both:
- **early stopping** inside a trial when its own validation score stops improving;
- **Optuna pruning** to stop a trial that is unpromising compared with other trials.

These are different mechanisms and you must explain the difference in the analysis.


In [ ]:
%pip -q install optuna

import optuna
from optuna.importance import get_param_importances

optuna.logging.set_verbosity(optuna.logging.WARNING)

SEARCH_EPOCHS = 6
EARLY_STOPPING_PATIENCE = 2
EARLY_STOPPING_MIN_DELTA = 1e-3


def build_optimizer(model, optimizer_name, lr, weight_decay, momentum=None):
    # TODO B7 — return AdamW or SGD according to optimizer_name.
    # Momentum must be used only for SGD.
    raise NotImplementedError


def objective(trial):
    # TODO B8 — define the search space.
    # Suggested ranges/categories:
    # base_channels: [16, 32, 48, 64]
    # dropout: 0.0 .. 0.5
    # use_batchnorm: [False, True]
    # activation: ["relu", "gelu"]
    # use_augmentation: [False, True]
    # optimizer: ["adamw", "sgd"]
    # lr: log scale
    # weight_decay: log scale
    # momentum only for SGD

    # TODO B9 — construct a fresh CIFAR10CNN and optimizer.

    # TODO B10 — choose the plain or augmented loader according to the trial.

    criterion = nn.CrossEntropyLoss()
    best_val_acc = -1.0
    best_epoch = -1
    epochs_without_improvement = 0
    early_stopped = False

    for epoch in range(SEARCH_EPOCHS):
        # TODO B11 — train one epoch and evaluate validation accuracy.
        # _, _ = run_epoch(...)
        # _, val_acc = run_epoch(...)

        # TODO B12 — report val_acc to Optuna.
        # trial.report(...)

        # TODO B13 — update early-stopping state using MIN_DELTA and PATIENCE.

        # TODO B14 — ask Optuna whether the trial should be pruned.
        # if trial.should_prune():
        #     raise optuna.TrialPruned()

        # TODO B15 — break when early stopping is triggered.
        pass

    trial.set_user_attr("best_epoch", best_epoch)
    trial.set_user_attr("early_stopped", early_stopped)
    return best_val_acc


sampler = optuna.samplers.TPESampler(seed=SEED)
pruner = optuna.pruners.MedianPruner(n_startup_trials=3, n_warmup_steps=1)
study = optuna.create_study(direction="maximize", sampler=sampler, pruner=pruner)

# TODO B16 — run a compact study. Start with 12–20 trials.
# study.optimize(objective, n_trials=...)


## B5 — Analyze the Optuna study

Do not stop at `study.best_params`.

Analyze:
1. the best validation score and configuration;
2. completed vs pruned trials;
3. how many completed trials were early-stopped;
4. parameter importance;
5. relationships between important hyperparameters and validation accuracy;
6. at least **two interactions** (for example optimizer × learning rate, capacity × dropout, or augmentation × model capacity).

Be cautious: Optuna importance values describe **this study and this search space**, not universal truths about CNNs.


In [ ]:
# TODO B17 — inspect the best trial.
# print("best value:", ...)
# print("best params:", ...)

# TODO B18 — create a DataFrame of all trials and inspect state/parameters.
# trials_df = study.trials_dataframe(...)
# display(...)

# TODO B19 — count COMPLETE, PRUNED, and early-stopped completed trials.

# TODO B20 — compute parameter importances.
# importances = get_param_importances(study)
# display(pd.Series(importances, name="importance"))

# OPTIONAL — use Optuna visualizations if useful in your discussion.
# optuna.visualization.plot_param_importances(study).show()
# optuna.visualization.plot_parallel_coordinate(study).show()


### B5 interpretation

Write **4–6 concise observations** based on your actual study. Include:
- which hyperparameters mattered most;
- whether the preferred optimizer required a particular learning-rate region;
- whether augmentation was generally selected by strong trials;
- evidence of underfitting/overfitting for small/large models;
- the difference between **early stopping** and **Optuna pruning** in your runs.


## B6 — Retrain the best configuration and use the test set once

After hyperparameters have been selected, build a **fresh model** from the best parameters.

For the final training run:
- use all 50,000 CIFAR-10 training images;
- use the augmentation decision selected by Optuna;
- train for approximately the best number of epochs observed during tuning;
- do not inspect the test set until training is complete.

Then evaluate the final model on the CIFAR-10 test set **once**.


In [ ]:
# TODO B21 — extract best hyperparameters and best_epoch from study.best_trial.
# best_params = ...
# FINAL_EPOCHS = ...

# TODO B22 — rebuild a fresh CIFAR10CNN using only model-related best_params.

# TODO B23 — rebuild the corresponding optimizer.

# Full 50k training dataset with the selected transform.
# full_train_dataset = cifar_train_aug_full if best_params["use_augmentation"] else cifar_train_plain_full
# full_train_loader = DataLoader(..., shuffle=True, ...)

# TODO B24 — train the fresh final model for FINAL_EPOCHS.

# TODO B25 — evaluate the test set ONCE and print test loss + accuracy.


## B7 — Save and verify the final model

Save enough information to reconstruct the experiment later. At minimum store:
- `state_dict`;
- selected hyperparameters;
- final number of epochs;
- test accuracy;
- CIFAR-10 normalization constants.

Then create a fresh model, reload the checkpoint, and verify that its test accuracy matches.


In [ ]:
FINAL_MODEL_PATH = "H2_cifar10_best_model.pt"

# TODO B26 — save a checkpoint dictionary with torch.save(...).

# TODO B27 — reconstruct a fresh model from the saved hyperparameters,
# load state_dict, move it to device, and evaluate it again.


## B8 — Final conclusions

Conclude the notebook with a compact comparison table containing at least:

| Experiment | Augmentation | Main regularization | Optimizer | Validation acc. | Test acc. |
|---|---|---|---|---:|---:|
| Initial CIFAR-10 baseline | ... | ... | ... | ... | — |
| Best Optuna configuration | ... | ... | ... | ... | ... |

Answer briefly:
1. Which changes most improved generalization?
2. Did the best configuration simply correspond to the largest model? Why/why not?
3. What did Optuna reveal that a one-factor-at-a-time experiment did not?
4. What would you try next if transfer learning were allowed? **Do not implement it in H2.**
